<a href="https://colab.research.google.com/github/databyhuseyn/DeepLearning/blob/main/Inception_Model_with_auxilary_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch.nn as nn
import torch

In [3]:
class InceptionBlock(nn.Module):

  def __init__(self, input_shape,
               onebyone_filter_size,
               pre_threebythree_filter_size,
               threebythree_filter_size,
               pre_fivebyfive_filter_size,
               fivebyfive_filter_size,
               post_maxpool_filter_size):

    super().__init__()

    self.conv_1by1 = nn.Conv2d(input_shape, onebyone_filter_size, kernel_size=(1, 1), stride=1)

    self.conv_pre_3by3 = nn.Conv2d(input_shape, pre_threebythree_filter_size, kernel_size=(1, 1), stride=1)
    self.conv_3by3 = nn.Conv2d(pre_threebythree_filter_size, threebythree_filter_size, kernel_size=(3, 3), stride=1, padding=1)

    self.conv_pre_5by5 = nn.Conv2d(input_shape, pre_fivebyfive_filter_size, kernel_size=(1, 1), stride=1)
    self.conv_5by5 = nn.Conv2d(pre_fivebyfive_filter_size, fivebyfive_filter_size, kernel_size=(5, 5), stride=1, padding=2)

    self.maxpool = nn.MaxPool2d(kernel_size=(3, 3), stride=1, padding=1)
    self.conv_maxpool_1by1 = nn.Conv2d(input_shape, post_maxpool_filter_size, kernel_size=(1, 1), stride=1)

    self.relu = nn.ReLU()

  def forward(self, X):
    one_by_one_output = self.relu(self.conv_1by1(X))

    pre_three_by_three_output = self.conv_pre_3by3(X)
    three_by_three_output = self.conv_3by3(self.relu(pre_three_by_three_output))

    pre_five_by_five_output = self.conv_pre_5by5(X)
    five_by_five_output = self.conv_5by5(self.relu(pre_five_by_five_output))

    max_pooled_output = self.maxpool(X)
    maxpooled_conv_output = self.conv_maxpool_1by1(self.relu(max_pooled_output))

    return torch.concat([one_by_one_output, three_by_three_output, five_by_five_output, maxpooled_conv_output], axis=1)

In [17]:
class Inception(nn.Module):
  def __init__(self, in_channels, out_channels):

    super().__init__()

    self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=(7, 7), stride=2, padding=3)
    self.maxpool1 = nn.MaxPool2d(kernel_size=(3, 3), stride=2, padding=1)
    self.conv_block = nn.Sequential(
        nn.Conv2d(64, 192, kernel_size=(1, 1)),
        nn.ReLU(),
        nn.Conv2d(192, 192, kernel_size=(3, 3), padding=1),
        nn.MaxPool2d(kernel_size=(3, 3), stride=2, padding=1)
    )

    self.inception_3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
    self.inception_3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64)

    self.maxpool2 = nn.MaxPool2d(kernel_size=(3, 3), stride=2, padding=1)

    self.inception_4a = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
    self.inception_4b = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
    self.inception_4c = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
    self.inception_4d = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
    self.inception_4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)

    self.maxpool3 = nn.MaxPool2d(kernel_size=(3, 3), stride=2, padding=1)

    self.inception_5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
    self.inception_5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

    self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
    self.flatten = nn.Flatten()
    self.dropout = nn.Dropout(0.4)
    self.output = nn.Linear(1024, out_channels)

    self.aux_output_1 = nn.Sequential(
        nn.Conv2d(in_channels=512, out_channels=64, kernel_size=(1, 1)),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, out_channels)
    )

    self.aux_output_2 = nn.Sequential(
        nn.Conv2d(in_channels=528, out_channels=64, kernel_size=(1, 1)),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, out_channels)
    )


  def forward(self, X):
    X = self.conv1(X)
    X = self.maxpool1(X)
    X = self.conv_block(X)
    X = self.inception_3a(X)
    X = self.inception_3b(X)
    X = self.maxpool2(X)
    out1 = self.inception_4a(X)
    X = self.inception_4b(out1)
    X = self.inception_4c(X)
    out2 = self.inception_4d(X)
    X = self.inception_4e(out2)
    X = self.maxpool3(X)
    X = self.inception_5a(X)
    X = self.inception_5b(X)
    X = self.global_avg_pool(X)
    X = self.flatten(X)
    X = self.dropout(X)
    out_main = self.output(X)
    aux_output_1 = self.aux_output_1(out1)
    aux_output_2 = self.aux_output_2(out2)

    return out_main, aux_output_1 , aux_output_2

In [18]:
model = Inception(3, 10)

In [19]:
model

Inception(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
  (maxpool1): MaxPool2d(kernel_size=(3, 3), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv_block): Sequential(
    (0): Conv2d(64, 192, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(192, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): MaxPool2d(kernel_size=(3, 3), stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (inception_3a): InceptionBlock(
    (conv_1by1): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
    (conv_pre_3by3): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
    (conv_3by3): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv_pre_5by5): Conv2d(192, 16, kernel_size=(1, 1), stride=(1, 1))
    (conv_5by5): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (maxpool): MaxPool2d(kernel_size=(3, 3), stride=1, padding=1, dilation=1, ceil_mode=False)
    (conv_maxpool_1by1): Conv2d(192, 

In [7]:
!pip install torchinfo

In [20]:
from torchinfo import summary

# Create a dummy input tensor to visualize the model
# The input_shape is (batch_size, channels, height, width)
# Assuming a common image size of 224x224 for a 3-channel input
dummy_input = torch.randn(1, 3, 224, 224)

# Display the model summary
summary(model, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
Inception                                [1, 10]                   --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,472
├─MaxPool2d: 1-2                         [1, 64, 56, 56]           --
├─Sequential: 1-3                        [1, 192, 28, 28]          --
│    └─Conv2d: 2-1                       [1, 192, 56, 56]          12,480
│    └─ReLU: 2-2                         [1, 192, 56, 56]          --
│    └─Conv2d: 2-3                       [1, 192, 56, 56]          331,968
│    └─MaxPool2d: 2-4                    [1, 192, 28, 28]          --
├─InceptionBlock: 1-4                    [1, 256, 28, 28]          --
│    └─Conv2d: 2-5                       [1, 64, 28, 28]           12,352
│    └─ReLU: 2-6                         [1, 64, 28, 28]           --
│    └─Conv2d: 2-7                       [1, 96, 28, 28]           18,528
│    └─ReLU: 2-8                         [1, 96, 28, 28]         

In [21]:
!pip install torchviz

In [23]:
from torchviz import make_dot

# Assuming `model` and `dummy_input` are already defined from previous cells
# Ensure model is in evaluation mode if dropout/batchnorm are present
model.eval()

# Perform a forward pass to get the computational graph
output = model(dummy_input)

# Create the visualization graph
# Set show_attrs=False to simplify the graph and prevent rendering issues
dot = make_dot(output,
               params=dict(model.named_parameters()),
               show_attrs=False)

# Set graph, node, and edge attributes for color and style
dot.graph_attr.update({
    'bgcolor': 'white',
    'rankdir': 'LR',
    'splines': 'ortho'
})
dot.node_attr.update({
    'style': 'filled',
    'fillcolor': 'lightgreen',
    'shape': 'box'
})
dot.edge_attr.update({
    'color': 'darkblue'
})

# Render and display the graph
dot.render('colorful_model_graph', format='png', cleanup=True) # Saves as colorful_model_graph.png

print("Colorful model graph saved as 'colorful_model_graph.png'. You can view it in the files sidebar.")

Colorful model graph saved as 'colorful_model_graph.png'. You can view it in the files sidebar.
